## pattern -18 Log Analysis

🟢 Level 1 — Basic log analysis

Q1. Count ERROR lines

Find how many lines contain ERROR.

In [36]:
from pathlib import Path

lines = Path("production.log").read_text().splitlines()

count = 0

for line in lines:
    if "ERROR" in line:
        count += 1
print(count)


5


Q2. Extract all ERROR lines

Return the complete error lines in a list.

In [37]:
from pathlib import Path

lines = Path("production.log").read_text().splitlines()

errors = []

for line in lines:
    if "ERROR" in lines:
        errors.append(line)
        
print(errors)


[]


Q3. Count each log level

Count INFO, ERROR, and WARNING.

In [38]:
from pathlib import Path

lines =  Path("production.log").read_text().splitlines()

frequency = {}

for line in lines:
    parts = line.split()
    level = parts[1]

    frequency[level] = frequency.get(level,0)+1

print(frequency)



{'ERROR': 5, 'INFO': 1, 'WARNING': 1}


Q4. Count errors by server

In [39]:
from pathlib import Path

lines = Path("production.log").read_text().splitlines()

errors_by_server = {}

for line in lines:
    parts = line.split()

    level = parts[1]
    server = parts[2]

    if level == "ERROR":
        errors_by_server[server] = errors_by_server.get(server, 0)+1
print(errors_by_server)


{'web01': 2, 'web03': 2, 'web02': 1}


Q5. Find the most problematic server

Find the server with the highest number of errors.

In [40]:
most_problematic_server = max(errors_by_server, key = errors_by_server.get)
print(most_problematic_server)

web01


🟡 Level 2 — Intermediate

Q6. Count unique error messages

Repeated messages should count only once in the unique-message collection.



In [41]:
from pathlib import Path

lines = Path("production.log").read_text().splitlines()

messages = set()

for line in lines:
    parts = line.split()

    if parts[1] == "ERROR":
        message = " ".join(parts[5:])
        messages.add(message)
print(messages)

{'Database failed', 'Gateway failed', 'Service unavailable'}


Q7. Find the most frequent error message

In [42]:
from pathlib import Path

lines = Path("production.log").read_text().splitlines()

frequency = {}

for line in lines:
    parts = line.split()
    if parts[1] == "ERROR":
        message = " ".join(parts[5:])
        frequency[message] = frequency.get(message,0)+1

most_common = max(frequency, key=frequency.get)
print(most_common)

Database failed


Q8. Sort servers by error count

In [43]:
sorted_servers = sorted(errors_by_server.items(), key = lambda x:x[1], reverse=True)
print(sorted_servers)

[('web01', 2), ('web03', 2), ('web02', 1)]


Q9. Create a reusable error-counting function

In [44]:
from pathlib import Path

def count_errors(filename):
    count = 0

    for line in Path(filename).read_text().splitlines():
        if "ERROR" in line:
            count += 1
    return count
print(count_errors("production.log"))


5


Q10. Search logs for a keyword

Find all lines containing database, ignoring case.

In [45]:
from pathlib import Path

def search_logs(filename,keyword):
    result = []

    for line in Path(filename).read_text().splitlines():
        if keyword.lower() in line.lower():
            result.append(line)
    return result

print(search_logs("production.log","database"))

['2026-09-29 ERROR web01 500 10.0.0.1 Database failed', '2026-09-29 ERROR web01 500 10.0.0.1 Database failed']


🟠 Level 3 — Regex and advanced patterns

For the next questions, use this log format:

Q11. Extract HTTP error codes using regex

In [46]:
from pathlib import Path
import re

text = Path("server.log").read_text()

codes = re.findall(r"\bERROR\s+\S+\s+(\d{3})\b",text)
print(codes)

['500', '503', '500']


Q12. Count each HTTP error code

In [ ]:
from pathlib import Path
import re

text = Path("production.log").read_text()

codes = re.findall(r"\bERROR\s+\S+\s+(\d{3})\b",text)

frequency = {}

for code in codes:
    frequency[code] = frequency.get(code,0)+1

print(frequency)

{'500': 2, '503': 2, '502': 1}


Q13. Extract IP addresses from logs

Assume log entries contain IP addresses.

In [51]:
from pathlib import Path
import re

text = Path("production.log").read_text()
'''2026-09-29 ERROR web03 503 10.0.0.3 Service unavailable'''
ips = re.findall(r"\b(?:\d{1,3}\.){3}\d{1,3}\b",text)

print(ips)



['10.0.0.1', '10.0.0.2', '10.0.0.1', '10.0.0.3', '10.0.0.2', '10.0.0.2', '10.0.0.3']


Q14. Find unique error records

Two records are duplicates only if their complete normalized text matches.

In [58]:
from pathlib import Path

lines = Path("production.log").read_text().splitlines()

seen = set()
unique_errors = []

for line in lines:
    line = line.strip()

    if "ERROR" not in line:
        continue

    if line not in seen:
        seen.add(line)
        unique_errors.append(line)

print(unique_errors)

['2026-09-29 ERROR web01 500 10.0.0.1 Database failed', '2026-09-29 ERROR web03 503 10.0.0.3 Service unavailable', '2026-09-29 ERROR web02 502 10.0.0.2 Gateway failed']


Q15. Find servers with more than one error

In [59]:
overloaded_servers = {server:count for server,count in errors_by_server.items() if count > 1}
print(overloaded_servers)

{'web01': 2, 'web03': 2}


🔴 Level 4 — Functions, JSON and production analysis

Q16. Build a complete log analyzer function

Return the total number of log entries, error count, and level frequency.

In [60]:
from pathlib import Path

def analyze_logs(filename):
    lines = [line for line in Path(filename).read_text().splitlines() if line.strip()]
    
    level_frequency = {}

    for line in lines:
        level = line.split()[2]
        level_frequency[level] = level_frequency.get(level,0)+1
  
    return {
        "total_logs": len(lines),
        "error_count": level_frequency.get("ERROR", 0),
        "level_frequency": level_frequency
    }

Q17. Convert a log report to JSON

In [61]:
import json

report = {
    "total_errors": 3,
    "errors_by_server": {
        "web02": 2,
        "web03": 1
    }  
}

json_text = json.dumps(report, indent = 4)
print(json_text)

{
    "total_errors": 3,
    "errors_by_server": {
        "web02": 2,
        "web03": 1
    }
}


Q18. Analyze logs already parsed into dictionaries

Suppose an API returns:

logs = [
    {"server": "web01", "level": "ERROR", "code": 500},
    {"server": "web02", "level": "INFO", "code": 200},
    {"server": "web01", "level": "ERROR", "code": 503}
]

Count errors by server.

In [1]:
logs = [
    {"server": "web01", "level": "ERROR", "code": 500},
    {"server": "web02", "level": "INFO", "code": 200},
    {"server": "web01", "level": "ERROR", "code": 503}
]

In [ ]:
error_by_server = {}

for log in logs:
    server = log["server"]
    if log["level"] == "ERROR":
        error_by_server[server] = (error_by_server.get(server,0)+1)
print(error_by_server)

{'web01': 2}


Q19. Process a large log file using a generator

In [3]:
def error_lines(filename):
    with open(filename,"r") as file:
        for line in file:
            if "ERROR" in line:
                yield line.strip()
for line in error_lines("server.log"):
    print(line)

2026-10-13 10:01:02 ERROR web02 500 Database failed
2026-10-13 10:02:03 ERROR web03 503 Connection timeout
2026-10-13 10:04:05 ERROR web02 500 Database failed


Q20. Handle missing files and invalid log lines

In [5]:
from pathlib import Path

def count_errors_safely(filename):
    count = 0
    try:
        with open(filename, "r") as file:
            for line in file:
                parts = line.split()

                if len(parts)<4:
                    continue

                if parts[2] == "ERROR":
                    count +=1

    except FileNotFoundError:
        print("log file not found")
        return None
    except PermissionError:
        print("permission denied")
        return None
    return count

print(count_errors_safely("server.log"))

3


🔥 Level 5 — DevOps and interview challenges

Q21. Analyze multiple log files recursively

In [ ]:
from pathlib import Path

def scan_log_files(directory):
    report = {}

    for file in Path(directory).rglob("*.log"):
        count = 0

        with file.open("r") as handle:
            for line in handle:
                if "ERROR" in line:
                    count += 1
        report[str(file)] = count

    return report

Q22. Find the top three servers by error count

In [7]:
top_servers = sorted(error_by_server.items(), key = lambda item:item[1], reverse = True)[:3]
print(top_servers)

[('web01', 2)]


Q23. Add logging to the analyzer

In [ ]:
import logging
from pathlib import Path

logging.basicConfig(
    filename = "analyzer.log",
    level = logging.INFO,
    format = "%(asctime)s %(levelname)s %(message)s"
)

def count_errors(filename):
    count = 0
    try :
        with Path(filename).open("r") as file:
            for line in file:
                if "ERROR" in line:
                    count += 1
        logging.info("processed %s; errors=%d", filename, count)

        return count
    except OSError:
        logging.exception("could not process %s", filename)
        return None


Q24. Analyze only production errors

Assume each parsed record contains environment, level, and server.

In [9]:
def production_errors(logs):
    frequency = {}

    for log in logs:
        if (log["environment"].lower() == "production"
            and log["level"].upper() == "ERROR"):
            server = log["server"].lower()
            frequency[server] = frequency.get(server,0)+1
    return frequency

Q25. Find unique error messages and their frequency

In [10]:
def error_message_frequency(logs):
    frequency = {}

    for log in logs:
        if log["level"].upper() == "ERROR":
            message = log["mressage"].strip().lower()

            frequency[message] = frequency.get(message,0)+1

    return frequency


Q26. Identify the most problematic server safely

In [11]:
def most_problematic_server(error_by_server):
    if not error_by_server:
        return None

    return max(error_by_server, key = error_by_server.get)


Q27. Detect repeated failures

Find error messages that occur at least three times.

In [12]:
def repeated_failures(logs):
    frequency = {}

    for log in logs:
        if log["level"].upper() == "ERROR":
            message = log["message"]

            frequency[message] = frequency.get(message,0)+1

    return {
        message : count
        for message,count in frequency.items()
        if count >= 3
    }


Q28. Calculate the error percentage

In [13]:
def error_percentage(logs):
    if not logs:
        return 0.0

    error_count = sum(
        1 for log in logs
        if log["level"].upper() == "ERROR"
    )

    return round(error_count/len(logs) * 100,2)

Q29. Generate a full production log report

In [ ]:
def production_log_report(logs):
    level_frequency = {}
    errors_by_server = {}
    error_code_frequency = {}
    unique_messages = set()


    for log in logs:
        level = log["level"].strip().upper()
        server = log["server"].strip().lower()

        level_frequency[level] = (level_frequency.get(level,0)+1)

        if level == "ERROR":
            errors_by_server[server] = (errors_by_server.get(server,0)+1)
            
            code = log["code"]
            error_code_frequency[code] = (error_code_frequency.get(code,0)+1)

            unique_messages.add(log["message"].strip())

    sorted_servers = sorted(errors_by_server.items(), key = lambda x:x[1], reverse=True)

    return {
        "total_logs": len(logs),
        "level_frequency": level_frequency,
        "total_errors": sum(errors_by_server.values()),
        "errors_by_server": errors_by_server,
        "error_code_frequency": error_code_frequency,
        "unique_error_messages": sorted(unique_messages),
        "most_problematic_server": (sorted_servers[0][0] if sorted_servers else None),
        "sorted_servers": sorted_servers
    }



Q30. Final cumulative challenge — production log analyzer

This combines File Processing, String Processing, Regex, Dictionaries, Sets, Functions, Frequency Analysis, Sorting, Exception Handling, Logging, and JSON.

Requirements:

Read a log file.

Skip blank or malformed lines.

Parse the timestamp, level, server, HTTP code, and message.

Count all valid log levels.

Count errors by server and HTTP code.

Identify unique error messages.

Find the most problematic server.

Sort servers by error count.

Save the report to JSON.

Handle file errors.

Assume each valid log line follows this format:

2026-10-13 10:01:02 ERROR web01 500 Database failed

In [ ]:
import json
import logging
import re
from pathlib import Path
from collections import Counter

logging.basicConfig(
    filename = "log_analyzer.log",
    levelname = logging.INFO
    format = "%(asctime)s %(levelname)s %(message)s"
)

LOG_PATTERN = re.compile(
    r"^(?P<date>\d{4}-\d{2}-\d{2})"
    r"(?P<time>\d{2}:\d{2}:\d{2})"
    r"(?P<level>INFO|WARNING|ERROR|DEBUG)"
    r"(?P<server>\S+)"
    r"(?P<code>\d{3})"
    r"(?P<message>.+)$"
)

def analyze_log_file(filename, report_filename = "report.json"):
    level_frequency = Counter()
    errors_by_server = Counter()
    error_code_frequency = Counter()
    unique_error_messages = set()
    valid_logs = 0

    try : 
        with Path(filename).open("r") as file:
            for line in file:
                line = line.strip()

                if not line :
                    continue

                match = LOG_PATTERN.match(line)

                if not match:
                    logging.warning("skipping malformed log line")
                    continue

                log = match.groupdict()
                valid_logs += 1

                level = log["level"]
                server = log["server"].lower()
                code = int(log["code"])
                message = log["message"]

                level_frequency[level] += 1

                if level == "ERROR":
                    errors_by_server[server] += 1
                    error_code_frequency[code] += 1
                    unique_error_messages.add(message)
                    
                    
        sorted_servers = sorted(errors_by_server.items(), key = lambda x:x[1], reverse = True)

        report = {
            "total_valid_logs": valid_logs,
            "level_frequency": dict(level_frequency),
            "total_errors": sum(errors_by_server.values()),
            "errors_by_server": dict(errors_by_server),
            "error_code_frequency": dict(error_code_frequency),
            "unique_error_messages": sorted(unique_error_messages),
            "most_problematic_server": (
                sorted_servers[0][0] if sorted_servers else None),
            "sorted_servers": sorted_servers
        }

        Path(report_filename).write_text(json.dumps(report, indent = 4))

        logging.info("Log report saved to %s", filename)
        
        return report

    except OSError:
        logging.exception("unable to read log/report file")
        return None

